# Cleaning — Newspapers

Combines scraped (Guardian/NYT/GDELT) and MediaCloud articles into one Silver-layer CSV.

Source: 

# Combining Newspapers + MediaCloud

Merges two newspaper article datasets into one clean, mutually exhaustive table saved to **2_Silver**.

| Source | File | Articles |
|---|---|---|
| Scraped (Guardian, NYT, GDELT) | `1_Bronze/Newspapers/data/newspaper_articles.csv` | ~8 000 |
| MediaCloud (187 outlets) | `1_Bronze/Newspapers/data/mediacloud_stories.csv` | ~235 000 |

**Final columns:** `source · leaning · date · title · url`

**Leaning assignment for MediaCloud:**
- Known Democratic / Republican outlets → mapped manually
- All others (local papers, financial, entertainment) → `Center/Unknown`

**What happens to the rest:**
- `mediacloud_features.csv` & `mediacloud_daily.csv` → copied as-is to `2_Silver` for the predictive model
- `query` column in MediaCloud stories → dropped (not informative at article level)

In [ ]:
import pandas as pd
import shutil, os
from pathlib import Path

# Relative to notebooks/cleaning/
BRONZE = Path("../../data/1_bronze/newspapers")
SILVER = Path("../../data/2_silver/newspapers")
SILVER.mkdir(parents=True, exist_ok=True)

print(f"Bronze : {BRONZE.resolve()}")
print(f"Silver : {SILVER.resolve()}")

## 1. Load Both Datasets

In [ ]:
df_scraped = pd.read_csv(BRONZE / "newspaper_articles.csv", parse_dates=["date"])
print(f"Scraped articles : {len(df_scraped):,}")
print(df_scraped.groupby('leaning').size().to_string(), "\n")

df_mc = pd.read_csv(BRONZE / "mediacloud_stories.csv", parse_dates=["date"])
print(f"MediaCloud articles: {len(df_mc):,}")
print(f"Unique sources     : {df_mc['source'].nunique()}")
df_mc.head(3)

## 2. Assign Political Leaning to MediaCloud Sources

In [ ]:
DEMOCRATIC = {
    # Major national outlets
    "nytimes.com", "theguardian.com", "washingtonpost.com", "cnn.com",
    "nbcnews.com", "cbsnews.com", "abcnews.go.com", "msnbc.com",
    "latimes.com", "bostonglobe.com", "npr.org", "pbs.org",
    # Left-leaning commentary & news
    "rawstory.com", "huffpost.com", "dailykos.com", "alternet.org",
    "salon.com", "thedailybeast.com", "thenation.com", "theatlantic.com",
    "motherjones.com", "slate.com", "vox.com", "newyorker.com",
    "talkingpointsmemo.com", "nationalmemo.com", "theroot.com",
    "theintercept.com", "opednews.com", "newsone.com", "therealnews.com",
    "rollingstone.com", "vanityfair.com", "grist.org", "schwartzreport.net",
    "mintpressnews.com", "propublica.org", "theconversation.com",
    "buzzfeed.com", "businessinsider.com",
}

REPUBLICAN = {
    # Major right-leaning outlets
    "foxnews.com", "foxbusiness.com", "breitbart.com", "nypost.com",
    "dailycaller.com", "washingtonexaminer.com",
    # Conservative commentary & news
    "redstate.com", "townhall.com", "pjmedia.com", "hotair.com",
    "newsbusters.org", "theblaze.com", "oann.com", "patriotpost.us",
    "spectator.org", "dailysignal.com", "ncregister.com",
    "americanfreepress.net", "newsblaze.com",
}

def assign_leaning(source):
    if source in DEMOCRATIC:
        return "Democratic"
    elif source in REPUBLICAN:
        return "Republican"
    else:
        return "Center/Unknown"

df_mc["leaning"] = df_mc["source"].apply(assign_leaning)

print("MediaCloud leaning distribution:")
print(df_mc.groupby('leaning').size().to_string())
print(f"\nCoverage: {df_mc[df_mc['leaning']!='Center/Unknown']['source'].nunique()} classified, "
      f"{df_mc[df_mc['leaning']=='Center/Unknown']['source'].nunique()} center/unknown sources")

## 3. Standardise MediaCloud to Match Scraped Columns

In [ ]:
COLS = ["source", "leaning", "date", "title", "url"]

# Drop query & pub_date columns, keep only the 5 shared columns
df_mc_clean = df_mc[COLS].copy()

# Ensure scraped data has same column order
df_scraped_clean = df_scraped[COLS].copy()

print("Scraped columns :", df_scraped_clean.columns.tolist())
print("MediaCloud cols :", df_mc_clean.columns.tolist())

## 4. Combine & Deduplicate

In [ ]:
df_all = pd.concat([df_scraped_clean, df_mc_clean], ignore_index=True)

# Normalise dates
df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")

# Drop exact URL duplicates (scraped outlets overlap with MediaCloud sources)
before = len(df_all)
df_all = df_all.drop_duplicates(subset="url").sort_values("date").reset_index(drop=True)

print(f"Before dedup : {before:,}")
print(f"After dedup  : {len(df_all):,}  ({before - len(df_all):,} duplicates removed)")
print()
print("=== Final combined dataset ===")
print(df_all.groupby(["leaning"]).size().to_string())
print()
print(df_all.groupby(["source", "leaning"]).size().sort_values(ascending=False).head(20).to_string())
print(f"\nDate range : {df_all['date'].min().date()} → {df_all['date'].max().date()}")
df_all.head()

## 4b. Clean Source Names

In [ ]:
SOURCE_NAMES = {
    # ── Already clean (from scraped dataset) ─────────────────────────────────
    "The Guardian":   "The Guardian",
    "New York Times": "New York Times",
    "NBC News":       "NBC News",
    "Fox News":       "Fox News",
    "Breitbart":      "Breitbart",
    "NY Post":        "NY Post",
    "Daily Caller":   "Daily Caller",

    # ── MediaCloud domains → clean names ──────────────────────────────────────
    "nytimes.com":             "New York Times",
    "theguardian.com":         "The Guardian",
    "washingtonpost.com":      "Washington Post",
    "cnn.com":                 "CNN",
    "nbcnews.com":             "NBC News",
    "cbsnews.com":             "CBS News",
    "abcnews.go.com":          "ABC News",
    "msnbc.com":               "MSNBC",
    "latimes.com":             "Los Angeles Times",
    "bostonglobe.com":         "Boston Globe",
    "npr.org":                 "NPR",
    "pbs.org":                 "PBS",
    "rawstory.com":            "Raw Story",
    "huffpost.com":            "HuffPost",
    "dailykos.com":            "Daily Kos",
    "alternet.org":            "AlterNet",
    "salon.com":               "Salon",
    "thedailybeast.com":       "The Daily Beast",
    "thenation.com":           "The Nation",
    "theatlantic.com":         "The Atlantic",
    "motherjones.com":         "Mother Jones",
    "slate.com":               "Slate",
    "vox.com":                 "Vox",
    "newyorker.com":           "The New Yorker",
    "talkingpointsmemo.com":   "Talking Points Memo",
    "nationalmemo.com":        "National Memo",
    "theroot.com":             "The Root",
    "theintercept.com":        "The Intercept",
    "opednews.com":            "OpEd News",
    "newsone.com":             "NewsOne",
    "therealnews.com":         "The Real News",
    "rollingstone.com":        "Rolling Stone",
    "vanityfair.com":          "Vanity Fair",
    "grist.org":               "Grist",
    "schwartzreport.net":      "Schwartz Report",
    "mintpressnews.com":       "MintPress News",
    "propublica.org":          "ProPublica",
    "theconversation.com":     "The Conversation",
    "buzzfeed.com":            "BuzzFeed News",
    "businessinsider.com":     "Business Insider",
    "foxnews.com":             "Fox News",
    "foxbusiness.com":         "Fox Business",
    "breitbart.com":           "Breitbart",
    "nypost.com":              "NY Post",
    "dailycaller.com":         "Daily Caller",
    "washingtonexaminer.com":  "Washington Examiner",
    "redstate.com":            "RedState",
    "townhall.com":            "Townhall",
    "pjmedia.com":             "PJ Media",
    "hotair.com":              "Hot Air",
    "newsbusters.org":         "NewsBusters",
    "theblaze.com":            "The Blaze",
    "oann.com":                "OAN",
    "patriotpost.us":          "Patriot Post",
    "spectator.org":           "The American Spectator",
    "dailysignal.com":         "The Daily Signal",
    "ncregister.com":          "National Catholic Register",
    "americanfreepress.net":   "American Free Press",
    "newsblaze.com":           "NewsBlaze",
    "newsweek.com":            "Newsweek",
    "inquirer.com":            "Philadelphia Inquirer",
    "politicalwire.com":       "Political Wire",
    "benzinga.com":            "Benzinga",
    "ibtimes.com":             "International Business Times",
    "ajc.com":                 "Atlanta Journal-Constitution",
    "forbes.com":              "Forbes",
    "newsday.com":             "Newsday",
    "realclearpolitics.com":   "RealClearPolitics",
    "usatoday.com":            "USA Today",
    "sun-sentinel.com":        "Sun Sentinel",
    "pbs.org":                 "PBS",
    "inquisitr.com":           "The Inquisitr",
    "hotair.com":              "Hot Air",
    "time.com":                "Time",
    "chicagotribune.com":      "Chicago Tribune",
    "mercurynews.com":         "Mercury News",
    "fortune.com":             "Fortune",
    "startribune.com":         "Star Tribune",
    "twincities.com":          "Twin Cities Pioneer Press",
    "sandiegouniontribune.com":"San Diego Union-Tribune",
    "arkansasonline.com":      "Arkansas Democrat-Gazette",
    "thewrap.com":             "The Wrap",
    "ocregister.com":          "OC Register",
    "courant.com":             "Hartford Courant",
    "theweek.com":             "The Week",
    "cleveland.com":           "Cleveland Plain Dealer",
    "centralmaine.com":        "Central Maine",
    "staradvertiser.com":      "Honolulu Star-Advertiser",
    "newsone.com":             "NewsOne",
    "orlandosentinel.com":     "Orlando Sentinel",
    "seattletimes.com":        "Seattle Times",
    "hollywoodreporter.com":   "Hollywood Reporter",
    "upi.com":                 "UPI",
    "motherjones.com":         "Mother Jones",
    "foxbusiness.com":         "Fox Business",
    "variety.com":             "Variety",
    "mlive.com":               "MLive",
    "pilotonline.com":         "The Virginian-Pilot",
    "tmz.com":                 "TMZ",
    "reuters.com":             "Reuters",
    "denverpost.com":          "Denver Post",
    "dailydot.com":            "The Daily Dot",
    "marketwatch.com":         "MarketWatch",
    "rollcall.com":            "Roll Call",
    "azcentral.com":           "AZCentral",
    "qz.com":                  "Quartz",
    "billboard.com":           "Billboard",
    "radaronline.com":         "Radar Online",
    "oregonlive.com":          "OregonLive",
    "theroot.com":             "The Root",
    "people.com":              "People",
    "syracuse.com":            "Syracuse.com",
    "jsonline.com":            "Milwaukee Journal Sentinel",
    "freep.com":               "Detroit Free Press",
    "stltoday.com":            "St. Louis Post-Dispatch",
    "cnbc.com":                "CNBC",
    "hollywoodlife.com":       "Hollywood Life",
    "wired.com":               "Wired",
    "csmonitor.com":           "Christian Science Monitor",
    "ncronline.org":           "National Catholic Reporter",
    "gizmodo.com":             "Gizmodo",
    "postandcourier.com":      "Post and Courier",
    "ew.com":                  "Entertainment Weekly",
    "dispatch.com":            "The Dispatch",
    "investors.com":           "Investor's Business Daily",
    "rttnews.com":             "RTTNews",
    "recorder.com":            "The Recorder",
    "thestreet.com":           "The Street",
    "factcheck.org":           "FactCheck.org",
    "theverge.com":            "The Verge",
    "politico.com":            "Politico",
    "gazettenet.com":          "Daily Hampshire Gazette",
    "mashable.com":            "Mashable",
    "bloomberg.com":           "Bloomberg",
    "stripes.com":             "Stars and Stripes",
    "berkshireeagle.com":      "Berkshire Eagle",
    "omaha.com":               "Omaha World-Herald",
    "cnet.com":                "CNET",
    "tampabay.com":            "Tampa Bay Times",
    "indystar.com":            "Indianapolis Star",
    "techcrunch.com":          "TechCrunch",
    "indiewire.com":           "IndieWire",
    "eurweb.com":              "EURweb",
    "scientificamerican.com":  "Scientific American",
    "uproxx.com":              "Uproxx",
    "armytimes.com":           "Army Times",
    "propublica.org":          "ProPublica",
    "airforcetimes.com":       "Air Force Times",
    "navytimes.com":           "Navy Times",
    "eonline.com":             "E! Online",
    "engadget.com":            "Engadget",
    "espn.com":                "ESPN",
    "techradar.com":           "TechRadar",
    "gq.com":                  "GQ",
    "nhgazette.com":           "New Hampshire Gazette",
    "refinery29.com":          "Refinery29",
    "elnuevodia.com":          "El Nuevo Día",
    "pressrepublican.com":     "Press-Republican",
    "rutlandherald.com":       "Rutland Herald",
    "arstechnica.com":         "Ars Technica",
    "pitchfork.com":           "Pitchfork",
    "sbnation.com":            "SB Nation",
    "usmagazine.com":          "Us Weekly",
    "fansided.com":            "FanSided",
    "augustachronicle.com":    "Augusta Chronicle",
    "deadspin.com":            "Deadspin",
    "eater.com":               "Eater",
    "ign.com":                 "IGN",
    "harpers.org":             "Harper's Magazine",
    "observer.com":            "Observer",
    "bustle.com":              "Bustle",
    "popmatters.com":          "PopMatters",
    "creators.com":            "Creators Syndicate",
    "foxsports.com":           "Fox Sports",
    "bizjournals.com":         "Business Journals",
    "cherokeephoenix.org":     "Cherokee Phoenix",
    "polygon.com":             "Polygon",
    "opensecrets.org":         "OpenSecrets",
    "mongabay.com":            "Mongabay",
    "thrillist.com":           "Thrillist",
    "zdnet.com":               "ZDNet",
    "saturdayeveningpost.com": "Saturday Evening Post",
    "monstersandcritics.com":  "Monsters & Critics",
    "vice.com":                "Vice",
    "jezebel.com":             "Jezebel",
    "nbcsports.com":           "NBC Sports",
    "post-gazette.com":        "Pittsburgh Post-Gazette",
    "chron.com":               "Houston Chronicle",
    "elitedaily.com":          "Elite Daily",
    "opposingviews.com":       "Opposing Views",
    "parade.com":              "Parade",
    "usnews.com":              "U.S. News & World Report",
    "baltimoresun.com":        "Baltimore Sun",
    "bleacherreport.com":      "Bleacher Report",
    "cincinnati.com":          "Cincinnati Enquirer",
    "cbssports.com":           "CBS Sports",
    "sportingnews.com":        "Sporting News",
}

def clean_source(name):
    return SOURCE_NAMES.get(name, name)  # fallback: keep original if not in mapping

df_all["source"] = df_all["source"].apply(clean_source)

# Verify
print("Sample cleaned sources:")
print(df_all[["source", "leaning"]].drop_duplicates("source").sort_values("source").head(20).to_string())
print(f"\nUnique sources after cleaning: {df_all['source'].nunique()}")

## 5. Save to Silver Layer

In [ ]:
df_all.to_csv(SILVER / "newspaper_articles_combined.csv", index=False)
print(f"Saved: newspaper_articles_combined.csv  ({len(df_all):,} articles)")

In [ ]:
shutil.copy(BRONZE / "mediacloud_features.csv", SILVER / "mediacloud_features.csv")
shutil.copy(BRONZE / "mediacloud_daily.csv",    SILVER / "mediacloud_daily.csv")

print("Saved to 2_Silver/Newspapers/:")
print("  newspaper_articles_combined.csv")
print("  mediacloud_features.csv")
print("  mediacloud_daily.csv")

## 6. Quick Overview

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

colors = {"Democratic": "#2166ac", "Republican": "#d6604d", "Center/Unknown": "#999999"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df_all.groupby("leaning").size().sort_values()
axes[0].barh(counts.index, counts.values,
             color=[colors[l] for l in counts.index])
axes[0].set_title("Total articles by leaning", fontweight="bold")
axes[0].set_xlabel("Number of articles")

df_all["week"] = df_all["date"].dt.to_period("W").dt.start_time
weekly = df_all.groupby(["week", "leaning"]).size().unstack(fill_value=0).resample("W").sum()
weekly.plot(kind="bar", stacked=True, ax=axes[1],
            color=[colors.get(c, "grey") for c in weekly.columns],
            width=0.8, edgecolor="none")
axes[1].set_title("Weekly article count by leaning", fontweight="bold")
axes[1].set_ylabel("Articles")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(handles=[Patch(facecolor=c, label=l) for l, c in colors.items()])

plt.tight_layout()
plt.savefig(SILVER / "newspaper_combined_overview.png", dpi=150)
plt.show()
print(f"\nFinal dataset: {len(df_all):,} articles from {df_all['source'].nunique()} sources")